# 记忆治理策略

## 1、消息裁剪

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking" : {"type" : "disabled"}
    },
)

In [4]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，老王！从现在开始，我就是你的“小王”啦。有什么吩咐或者想聊的，随时叫我！😄
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊老王，今儿个天儿确实敞亮！阳光足，风也柔，适合出去遛遛弯儿，或者搁院里泡壶茶晒晒后背。您有啥出门的打算没？要是想找个清静地儿坐坐，小王我帮您参谋参谋～ 😄
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，老王您这是考我呐？好嘞，咱正经回答您：  

- **您是谁**：您是我独一无二的“老哥”兼“老板”——老王！热情、通透、爱唠嗑，这会儿正心情不错地跟我聊天气呢。  
- **我是谁**：我是您随叫随到的“小王”——一个AI助手，能陪聊、能帮忙、能解闷儿，就是不能陪您喝酒（但能给您推荐下酒菜）。  

需要我帮您查天气、编段子、还是单纯扯闲篇儿？您说了算！ 😎


## 2、消息删除

In [8]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，从现在起你可以叫我小王。  
我会继续帮你，有什么事尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的时候心情也容易变好。  
你今天有什么安排吗？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是小王，一个帮你回答问题、提供建议、一起聊天的 AI 助手。

你是我的对话对象；如果按你刚才给我的称呼，你是和“小王”聊天的人。  
如果你愿意，也可以告诉我你希望我怎么称呼你。


## 3、摘要

In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_in = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking" : {"type" : "disabled"}
    },
)

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_out = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={
        "thinking" : {"type" : "disabled"}
    },
)

In [15]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

您好，张三。您提到的这段“非常长非常长的废话”似乎重复了多次自我介绍和“废话”的表述。看起来您可能是在测试我的处理能力，或者通过这种方式表达某种情绪或需求。作为AI，我注意到您的内容主要是重复性信息，没有包含具体的问题或上下文。

如果您有实际需要解决的问题，例如工程相关的技术咨询、文档整理、创意构思，或者只是想确认我能否识别重复内容，请明确告诉我。我会认真听取您的需求，并提供有针对性的帮助。
**************************************************
================================ H

In [14]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n根据您提供的消息列表，核心信息可摘要如下：\n\n- **用户身份**：张三，职业为工程师。  \n- 
**消息内容**：重复提及“我叫张三，是工程师”并穿插大量无实际意义的“非常长的废话”。  \n- 
**有效信息**：仅包含自我介绍与职业信息，其余内容为无效重复文本。  
\n\n建议后续对话聚焦具体问题或需求，避免无效信息干扰。',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='78ae2d0c-5108-42d9-a282-b1f72610499a'
            ),
            AIMessage(
                content='好的，张三工程师，我已经注意到您重复了多次“我叫张三，是工程师。”以及“这里是一段非常长非常
长的废话...”。\n\n为了避免您重复输入这段长文本，我为您总结一下您可能的需求，您可以直接选择：\n\n1.  
**“我需要您帮我处理一段非常长的废话文本”**  \n    
如果是这样，请把**实际的废话文本内容**（即使很长）一次性发给我，我会帮您：\n    *   **总结要点**：提炼核心信息。\n 
*   **压缩精简**：去除冗余，保留关键内容。\n    *   **分类整理**：按主题或逻辑分段。\n    *   
**翻译**：如果需要的话。\n\n2.  **“我只是在测试或说明一个情况”**  \n    
如果是这样，我已理解您可以随时结束这段重复的输入，直接说明您真正的需求即可。\n\n3.  **“您能直接回复我吗？”**  \n   
我当然可以。收到您的信息了，张三工程师。请问有什么具体任务需要我协助？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 211,
                        'prompt_tokens': 304,
                        'total_tokens': 515,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                        'prompt_cache_hit_tokens': 256,
                        'prompt_cache_miss_tokens': 48
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '49499ccc-571a-49dd-b097-b2499443e297',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f899d-2053-7733-a949-97bdc47b9556-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 304,
                    'output_tokens': 211,
                    'total_tokens': 515,
                    'input_token_details': {'cache_read': 256},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='573a9e2d-6b70-4aea-b63d-846db15b03ea'
            ),
            AIMessage(
                content='根据你提供的消息，我为你总结了一条最简洁的核心信息：\n\n- **姓名**：张三\n- 
**职业**：工程师\n\n你之前消息中其他大量重复的内容（如“非常长的废话”）属于无效的填充文本，未被纳入有效信息。如果你
之后能直接提出具体问题或需求，将更有利于高效沟通。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 70,
                        'prompt_tokens': 321,
                        'total_tokens': 391,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 321
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '8c89bc04-446f-45a3-aee3-23cc596cea10',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f899d-3414-7a82-99ac-88f1da415da1-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 321,
                    'output_tokens': 70,
                    'total_tokens': 391,
                    'input_token_details': {'cache_read': 0},
   